# CWRU 10-Class MLR — PYNQ AXI4-Stream Test (Master Experiment Data)


In [1]:
from pathlib import Path
import time
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

BIT_FILE='logistic_fault.bit'
HWH_FILE='logistic_fault.hwh'

# One master archive created by the training notebook.
DATA_FILE='experiment_data.npz'

# ------------------------------------------------------------
# PYNQ experiment selector.
# Match these two values to the MODEL_W<window>_B<bits> #define
# used when the current HLS/Vivado bitstream was built.
# ------------------------------------------------------------
TEST_WINDOW_SIZE=512
TEST_SAMPLE_BITS=12
CFG=f'w{TEST_WINDOW_SIZE}_b{TEST_SAMPLE_BITS}'

NUM_FEATURES=3
NUM_CLASSES=10
MAX_WINDOWS=9600
SUPPORTED_WINDOW_SIZES=(256,512,1024,2048)
SUPPORTED_RAW_BITS=(4,8,12,16)

NUM_INPUT_BUSES=4
BUS_BITS=64

print('Selected PYNQ configuration:',CFG)
print('Expected matching HLS macro:',
      f'MODEL_W{TEST_WINDOW_SIZE}_B{TEST_SAMPLE_BITS}')

for n in [BIT_FILE,HWH_FILE,DATA_FILE]:
    print(n,Path(n).exists())


Selected PYNQ configuration: w512_b12
Expected matching HLS macro: MODEL_W512_B12
logistic_fault.bit True
logistic_fault.hwh True
experiment_data.npz True


In [2]:
def accuracy_score(y_true,y_pred):
    return float(np.mean(np.asarray(y_true)==np.asarray(y_pred)))

def confusion_matrix(y_true,y_pred,labels):
    labels=np.asarray(labels); lookup={int(v):i for i,v in enumerate(labels)}
    cm=np.zeros((len(labels),len(labels)),dtype=np.int64)
    for t,p in zip(np.asarray(y_true),np.asarray(y_pred)):
        if int(t) in lookup and int(p) in lookup: cm[lookup[int(t)],lookup[int(p)]]+=1
    return cm

def classification_report(y_true,y_pred,labels,target_names,digits=5):
    cm=confusion_matrix(y_true,y_pred,labels); rows=[]; P=[];R=[];F=[];S=[]
    width=max(12,max(len(str(n)) for n in target_names)+2)
    rows += [f"{'':<{width}} {'precision':>10} {'recall':>10} {'f1-score':>10} {'support':>9}",'']
    for i,name in enumerate(target_names):
        tp=cm[i,i]; sup=cm[i,:].sum(); pr=cm[:,i].sum()
        p=tp/pr if pr else 0.; r=tp/sup if sup else 0.; f=2*p*r/(p+r) if p+r else 0.
        P.append(p);R.append(r);F.append(f);S.append(sup)
        rows.append(f'{name:<{width}} {p:10.{digits}f} {r:10.{digits}f} {f:10.{digits}f} {int(sup):9d}')
    total=int(np.sum(S)); acc=accuracy_score(y_true,y_pred)
    rows += ['',f"{'accuracy':<{width}} {'':>10} {'':>10} {acc:10.{digits}f} {total:9d}"]
    rows.append(f"{'macro avg':<{width}} {np.mean(P):10.{digits}f} {np.mean(R):10.{digits}f} {np.mean(F):10.{digits}f} {total:9d}")
    rows.append(f"{'weighted avg':<{width}} {np.average(P,weights=S):10.{digits}f} {np.average(R,weights=S):10.{digits}f} {np.average(F,weights=S):10.{digits}f} {total:9d}")
    return '\n'.join(rows)

def classification_metrics(y_true,y_pred,labels,target_names):
    """
    Return per-class, macro, weighted, and balanced multiclass metrics.
    """
    labels=np.asarray(labels)
    cm=confusion_matrix(y_true,y_pred,labels)

    tp=np.diag(cm).astype(np.float64)
    support=cm.sum(axis=1).astype(np.float64)
    predicted=cm.sum(axis=0).astype(np.float64)

    precision=np.divide(
        tp,predicted,
        out=np.zeros_like(tp),
        where=predicted!=0
    )
    recall=np.divide(
        tp,support,
        out=np.zeros_like(tp),
        where=support!=0
    )
    f1=np.divide(
        2*precision*recall,
        precision+recall,
        out=np.zeros_like(tp),
        where=(precision+recall)!=0
    )

    total=float(np.sum(support))
    weights=(support/total) if total else np.zeros_like(support)

    per_class=pd.DataFrame({
        'class':list(target_names),
        'precision':precision,
        'recall':recall,
        'f1_score':f1,
        'support':support.astype(np.int64),
    })

    summary={
        'balanced_accuracy':float(np.mean(recall)),
        'macro_precision':float(np.mean(precision)),
        'macro_recall':float(np.mean(recall)),
        'macro_f1':float(np.mean(f1)),
        'weighted_precision':float(np.sum(weights*precision)),
        'weighted_recall':float(np.sum(weights*recall)),
        'weighted_f1':float(np.sum(weights*f1)),
    }

    return summary,per_class


def fault_detection_metrics(y_true,y_pred,normal_class=0):
    """
    Collapse the 10-class problem into Normal vs Fault.

    Any ground-truth class other than normal_class is an actual fault.
    Any predicted class other than normal_class is a detected fault.

    This measures fault detection independent of exact fault subtype.
    """
    y_true=np.asarray(y_true)
    y_pred=np.asarray(y_pred)

    actual_fault=(y_true!=normal_class)
    predicted_fault=(y_pred!=normal_class)

    tp=int(np.sum(actual_fault & predicted_fault))
    fp=int(np.sum(~actual_fault & predicted_fault))
    fn=int(np.sum(actual_fault & ~predicted_fault))
    tn=int(np.sum(~actual_fault & ~predicted_fault))

    precision=tp/(tp+fp) if (tp+fp) else 0.0
    recall=tp/(tp+fn) if (tp+fn) else 0.0
    f1=(
        2*precision*recall/(precision+recall)
        if (precision+recall)
        else 0.0
    )

    return {
        'fault_precision':float(precision),
        'fault_recall':float(recall),
        'fault_f1':float(f1),
        'fault_tp':tp,
        'fault_fp':fp,
        'fault_fn':fn,
        'fault_tn':tn,
    }



## Load overlay, HLS IP, and AXI DMA


In [3]:
overlay=Overlay(BIT_FILE)
print('IPs:',list(overlay.ip_dict.keys()))

# Locate the HLS accelerator.
name='logistic_fault_predict_0'
if name not in overlay.ip_dict:
    cand=[x for x in overlay.ip_dict if 'logistic' in x.lower()]
    if not cand:
        raise RuntimeError('No logistic HLS IP found')
    name=cand[0]

ip=getattr(overlay,name)

# ------------------------------------------------------------
# Locate FOUR AXI DMA blocks.
# Each DMA supplies one 64-bit input stream.
# ------------------------------------------------------------

def _natural_key(s):
    return [
        int(x) if x.isdigit() else x.lower()
        for x in re.split(r'(\d+)',s)
    ]

dma_names=sorted(
    [x for x in overlay.ip_dict if 'dma' in x.lower()],
    key=_natural_key
)

if len(dma_names)<4:
    raise RuntimeError(
        'Four-bus HLS design requires FOUR AXI DMA MM2S blocks. '
        f'Found only: {dma_names}'
    )

# If there are more than four DMA-like IPs, explicitly edit this list.
dma_names=dma_names[:4]
dmas=[getattr(overlay,n) for n in dma_names]

print('Using HLS IP:',name)
print('Using DMAs:',dma_names)
print(ip.register_map)

# None of the four AXI4-Stream inputs should appear as AXI-Lite pointer regs.


IPs: ['logistic_fault_predi_0', 'axi_dma_0', 'axi_dma_1', 'axi_dma_2', 'axi_dma_3', 'processing_system7_0']
Using HLS IP: logistic_fault_predi_0
Using DMAs: ['axi_dma_0', 'axi_dma_1', 'axi_dma_2', 'axi_dma_3']
RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0),
  predictions_1 = Register(predictions=write-only),
  predictions_2 = Register(predictions=write-only),
  window_count = Register(window_count=write-only),
  window_size = Register(window_size=write-only)
}


## Load the selected experiment from `experiment_data.npz`


In [4]:
d=np.load(DATA_FILE,allow_pickle=False)

available_configs=[str(x) for x in d['available_configs']]
print('Available configurations:',available_configs)

if CFG not in available_configs:
    raise KeyError(
        f'{CFG} is not present in {DATA_FILE}. '
        'Run the corresponding training experiment and rebuild the master archive.'
    )

def dk(field):
    return f'{CFG}__{field}'

def scalar_int(field):
    return int(np.asarray(d[dk(field)]).reshape(-1)[0])

windows=d[dk('windows')].astype(np.int16)
exported_features=d[dk('raw_features')].astype(np.float32)
X_test_scaled=d[dk('scaled_features')].astype(np.float32)
y_test=d[dk('y_test')].astype(np.int32)

WINDOW_SIZE=scalar_int('window_size')
RAW_SAMPLE_BITS=scalar_int('raw_sample_bits')
ZC_EPSILON=scalar_int('zc_epsilon')
FEATURE_TOTAL_BITS=scalar_int('feature_total_bits')
FEATURE_INT_BITS=scalar_int('feature_int_bits')
FEATURE_FRAC_BITS=FEATURE_TOTAL_BITS-FEATURE_INT_BITS

# The selected master-data entry must match the selector at the top.
assert WINDOW_SIZE==TEST_WINDOW_SIZE
assert RAW_SAMPLE_BITS==TEST_SAMPLE_BITS

assert WINDOW_SIZE in SUPPORTED_WINDOW_SIZES
assert RAW_SAMPLE_BITS in SUPPORTED_RAW_BITS

SAMPLES_PER_BUS=BUS_BITS//RAW_SAMPLE_BITS
SAMPLES_PER_PACKET=NUM_INPUT_BUSES*SAMPLES_PER_BUS
PACKETS_PER_WINDOW=(
    WINDOW_SIZE+SAMPLES_PER_PACKET-1
)//SAMPLES_PER_PACKET
LAST_PACKET_VALID_SAMPLES=(
    WINDOW_SIZE-(PACKETS_PER_WINDOW-1)*SAMPLES_PER_PACKET
)
BYTES_PER_BUS_PER_WINDOW=(
    PACKETS_PER_WINDOW*np.dtype(np.uint64).itemsize
)

# Verify that the master archive packing metadata matches what this
# notebook independently derives from the selected configuration.
assert scalar_int('bus_bits')==BUS_BITS
assert scalar_int('num_input_buses')==NUM_INPUT_BUSES
assert scalar_int('samples_per_bus')==SAMPLES_PER_BUS
assert scalar_int('samples_per_packet')==SAMPLES_PER_PACKET
assert scalar_int('packets_per_window')==PACKETS_PER_WINDOW
assert scalar_int('last_packet_valid_samples')==LAST_PACKET_VALID_SAMPLES

mean=d[dk('mean')].astype(np.float32)
scale=d[dk('scale')].astype(np.float32)
W=d[dk('weights')].astype(np.float32)
B=d[dk('bias')].astype(np.float32)
W_hw=d[dk('hw_weights')].astype(np.float32)
B_hw=d[dk('hw_bias')].astype(np.float32)
W_hw_q18=d[dk('hw_weights_q18')].astype(np.float32)
B_hw_q18=d[dk('hw_bias_q18')].astype(np.float32)

class_names=[str(x) for x in d[dk('class_names')]]
feature_names=[str(x) for x in d[dk('feature_names')]]

assert windows.ndim==2 and windows.shape[1]==WINDOW_SIZE
assert exported_features.shape==(len(windows),NUM_FEATURES)
assert X_test_scaled.shape==(len(windows),NUM_FEATURES)
assert len(y_test)==len(windows)

window_count=len(y_test)
assert window_count<=MAX_WINDOWS

print('Configuration key:',CFG)
print('Window size:',WINDOW_SIZE)
print('Raw sample bits:',RAW_SAMPLE_BITS)
print('ZC epsilon:',ZC_EPSILON)
print('Feature format:',f'Q{FEATURE_INT_BITS}.{FEATURE_FRAC_BITS}')
print('Samples/bus:',SAMPLES_PER_BUS)
print('Samples/logical packet:',SAMPLES_PER_PACKET)
print('Packets/window:',PACKETS_PER_WINDOW)
print('Last packet valid samples:',LAST_PACKET_VALID_SAMPLES)
print('Bus utilization:',SAMPLES_PER_BUS*RAW_SAMPLE_BITS,'/ 64 bits')
print('Bytes/bus/window:',BYTES_PER_BUS_PER_WINDOW)
print('Total bus bytes/window:',BYTES_PER_BUS_PER_WINDOW*NUM_INPUT_BUSES)
print('Windows:',windows.shape,windows.dtype)


Available configurations: ['w256_b4', 'w256_b8', 'w256_b12', 'w256_b16', 'w512_b4', 'w512_b8', 'w512_b12', 'w512_b16', 'w1024_b4', 'w1024_b8', 'w1024_b12', 'w1024_b16', 'w2048_b4', 'w2048_b8', 'w2048_b12', 'w2048_b16']
Configuration key: w512_b12
Window size: 512
Raw sample bits: 12
ZC epsilon: 16
Feature format: Q12.6
Samples/bus: 5
Samples/logical packet: 20
Packets/window: 26
Last packet valid samples: 12
Bus utilization: 60 / 64 bits
Bytes/bus/window: 208
Total bus bytes/window: 832
Windows: (6000, 512) int16


## Software end-to-end reference from the raw int8 windows



In [5]:
def extract_features_numpy(w):
    w=np.asarray(w,dtype=np.int16)
    wi=w.astype(np.int32)

    peak=wi.max(axis=1).astype(np.float32)

    shift=int(np.log2(WINDOW_SIZE))
    assert (1<<shift)==WINDOW_SIZE

    mav_sum=np.abs(wi.astype(np.int64)).sum(axis=1,dtype=np.int64)
    mav_encoded=mav_sum >> (shift-FEATURE_FRAC_BITS)
    mav=(
        mav_encoded.astype(np.float64) /
        (1<<FEATURE_FRAC_BITS)
    ).astype(np.float32)

    a=wi[:,:-1]
    b=wi[:,1:]
    sign=((a<0)&(b>0))|((a>0)&(b<0))
    delta=np.abs(b.astype(np.int64)-a.astype(np.int64))
    zc=np.count_nonzero(
        sign & (delta>ZC_EPSILON),
        axis=1
    ).astype(np.float32)

    return np.column_stack([peak,mav,zc]).astype(np.float32)

t0=time.perf_counter()
software_features=extract_features_numpy(windows)

software_logits=((software_features-mean)/scale)@W.T+B
software_predictions=np.argmax(software_logits,axis=1).astype(np.int32)

folded_logits=software_features@W_hw.T+B_hw
folded_predictions=np.argmax(folded_logits,axis=1).astype(np.int32)

q18_logits=software_features@W_hw_q18.T+B_hw_q18
q18_predictions=np.argmax(q18_logits,axis=1).astype(np.int32)

software_end2end_s=time.perf_counter()-t0

print('Max feature difference vs training export:',
      float(np.max(np.abs(software_features-exported_features))))
print('Folded/original agreement:',
      float(np.mean(folded_predictions==software_predictions)))
print('Q18/float-folded agreement:',
      float(np.mean(q18_predictions==folded_predictions)))
print(f'Q18 software-reference accuracy: {np.mean(q18_predictions==y_test):.6f}')
print(f'Software end-to-end average: {software_end2end_s/window_count*1e6:.6f} us/window')


Max feature difference vs training export: 0.0
Folded/original agreement: 1.0
Q18/float-folded agreement: 0.9991666666666666
Q18 software-reference accuracy: 0.890500
Software end-to-end average: 346.666735 us/window


## Max-pack samples into four 64-bit AXI4-Stream inputs



In [6]:
def pack_windows_to_four_u64(w, sample_bits):
    """
    Max-pack four physical 64-bit AXI streams.

    4-bit  -> 16 samples/bus -> 64 samples/logical packet
    8-bit  ->  8 samples/bus -> 32 samples/logical packet
    12-bit ->  5 samples/bus -> 20 samples/logical packet
    16-bit ->  4 samples/bus -> 16 samples/logical packet

    The final logical packet is zero-padded when the window is not an exact
    multiple of SAMPLES_PER_PACKET. The HLS design ignores padded samples.
    """
    w=np.ascontiguousarray(w,dtype=np.int16)

    assert w.ndim==2
    assert sample_bits in SUPPORTED_RAW_BITS

    samples_per_bus=BUS_BITS//sample_bits
    samples_per_packet=NUM_INPUT_BUSES*samples_per_bus
    packets_per_window=(
        w.shape[1]+samples_per_packet-1
    )//samples_per_packet

    assert samples_per_bus*sample_bits <= BUS_BITS

    padded_samples=packets_per_window*samples_per_packet
    padded=np.zeros(
        (w.shape[0],padded_samples),
        dtype=np.int16
    )
    padded[:,:w.shape[1]]=w

    groups=padded.reshape(
        w.shape[0],
        packets_per_window,
        samples_per_packet
    ).astype(np.int64)

    mask=np.uint64((1<<sample_bits)-1)
    packed_buses=[]

    for bus in range(NUM_INPUT_BUSES):
        start=bus*samples_per_bus
        stop=start+samples_per_bus
        lane=groups[:,:,start:stop]

        # Signed value -> low sample_bits of two's-complement representation.
        lane_u=(lane.astype(np.uint64) & mask)

        packed=np.zeros(lane_u.shape[:2],dtype=np.uint64)

        for j in range(samples_per_bus):
            packed |= lane_u[:,:,j] << np.uint64(j*sample_bits)

        packed_buses.append(packed)

    return packed_buses


packed_buses=pack_windows_to_four_u64(
    windows,
    RAW_SAMPLE_BITS
)

for b,packed in enumerate(packed_buses):
    assert packed.shape==(window_count,PACKETS_PER_WINDOW)
    print(f'bus{b} packed shape:',packed.shape,packed.dtype)

input_buffers=[
    allocate(
        (window_count*PACKETS_PER_WINDOW,),
        dtype=np.uint64
    )
    for _ in range(NUM_INPUT_BUSES)
]

for b in range(NUM_INPUT_BUSES):
    input_buffers[b][:]=packed_buses[b].reshape(-1)

prediction_buffer=allocate((window_count,),dtype=np.int32)
prediction_buffer[:]=0

CTRL=0x00
AP_START=0x01
AP_DONE=0x02
AP_IDLE=0x04
AP_READY=0x08

reg_meta=overlay.ip_dict[name].get('registers',{})

def find_reg_offset(token,fallback):
    token=token.lower()
    for reg_name,meta in reg_meta.items():
        if token in reg_name.lower():
            off=meta.get('address_offset',fallback)
            return int(off,0) if isinstance(off,str) else int(off)
    return fallback

PRED_LO=find_reg_offset('predictions',0x10)
PRED_HI=PRED_LO+4
COUNT=find_reg_offset('window_count',0x1C)
WINDOW_SIZE_REG=find_reg_offset('window_size',0x24)

def write_addr(lo,hi,a):
    a=int(a)
    ip.write(lo,a & 0xffffffff)
    ip.write(hi,(a>>32)&0xffffffff)

write_addr(PRED_LO,PRED_HI,prediction_buffer.physical_address)

for b,buf in enumerate(input_buffers):
    print(
        f'DMA{b} source buffer:',
        hex(buf.physical_address),
        buf.dtype,
        buf.shape
    )

print('Prediction buffer:',
      hex(prediction_buffer.physical_address),
      prediction_buffer.dtype,
      prediction_buffer.shape)


bus0 packed shape: (6000, 26) uint64
bus1 packed shape: (6000, 26) uint64
bus2 packed shape: (6000, 26) uint64
bus3 packed shape: (6000, 26) uint64
DMA0 source buffer: 0x15900000 uint64 (156000,)
DMA1 source buffer: 0x15b00000 uint64 (156000,)
DMA2 source buffer: 0x15d00000 uint64 (156000,)
DMA3 source buffer: 0x15f00000 uint64 (156000,)
Prediction buffer: 0x15850000 int32 (6000,)


In [7]:
print(list(overlay.ip_dict.keys()))
print(ip.register_map)

['logistic_fault_predi_0', 'axi_dma_0', 'axi_dma_1', 'axi_dma_2', 'axi_dma_3', 'processing_system7_0']
RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0),
  predictions_1 = Register(predictions=write-only),
  predictions_2 = Register(predictions=write-only),
  window_count = Register(window_count=write-only),
  window_size = Register(window_size=write-only)
}


In [8]:
def decode_ctrl(v):
    return {
        'raw':hex(v),
        'start':bool(v & AP_START),
        'done':bool(v & AP_DONE),
        'idle':bool(v & AP_IDLE),
        'ready':bool(v & AP_READY),
    }

def program_hls_args(n):
    write_addr(PRED_LO,PRED_HI,prediction_buffer.physical_address)
    ip.write(COUNT,int(n))
    ip.write(WINDOW_SIZE_REG,int(WINDOW_SIZE))

def dma_idle_list():
    return [dma.sendchannel.idle for dma in dmas]

def run_debug(n=1,timeout_s=2.0):
    assert 1<=n<=window_count

    prediction_buffer[:n]=-999

    for buf in input_buffers:
        buf.sync_to_device()

    prediction_buffer.sync_to_device()

    program_hls_args(n)

    nbytes_per_bus=int(
        n *
        PACKETS_PER_WINDOW *
        np.dtype(np.uint64).itemsize
    )

    print('Testing:')
    print('  windows       =',n)
    print('  window_size   =',WINDOW_SIZE)
    print('  packets/window=',PACKETS_PER_WINDOW)
    print('  bytes/bus     =',nbytes_per_bus)
    print('  total bytes   =',nbytes_per_bus*NUM_INPUT_BUSES)
    print('  pred addr     =',hex(prediction_buffer.physical_address))
    print('CTRL before start:',decode_ctrl(ip.read(CTRL)))
    print('DMA idle before:',dma_idle_list())

    # Start the HLS consumers first.
    ip.write(CTRL,AP_START)

    # Then launch all four MM2S channels. transfer() is non-blocking.
    for dma,buf in zip(dmas,input_buffers):
        dma.sendchannel.transfer(
            buf,
            nbytes=nbytes_per_bus
        )

    t0=time.perf_counter()

    while True:
        ctrl=ip.read(CTRL)

        if ctrl & AP_DONE:
            elapsed=time.perf_counter()-t0

            for dma in dmas:
                dma.sendchannel.wait()

            print('DONE after:',elapsed,'seconds')
            print('CTRL:',decode_ctrl(ctrl))
            print('DMA idle:',dma_idle_list())

            prediction_buffer.sync_from_device()

            print(
                'First predictions:',
                np.asarray(prediction_buffer[:min(n,10)])
            )

            return elapsed

        if time.perf_counter()-t0>timeout_s:
            print('TIMEOUT')
            print('CTRL:',decode_ctrl(ctrl))
            print('DMA idle:',dma_idle_list())

            # Raw MM2S status registers, useful for identifying
            # which lane is blocked or in error.
            for i,dma in enumerate(dmas):
                try:
                    print(f'DMA{i} MM2S_DMASR:',hex(dma.mmio.read(0x04)))
                except Exception as e:
                    print(f'DMA{i} status read failed:',e)

            prediction_buffer.sync_from_device()
            print('Prediction[0]:',int(prediction_buffer[0]))

            raise TimeoutError(
                'Four-bus FPGA did not finish. Reload overlay before retrying.'
            )


In [9]:
run_debug(1)
# After this passes, try:
# run_debug(5)


Testing:
  windows       = 1
  window_size   = 512
  packets/window= 26
  bytes/bus     = 208
  total bytes   = 832
  pred addr     = 0x15850000
CTRL before start: {'raw': '0x4', 'start': False, 'done': False, 'idle': True, 'ready': False}
DMA idle before: [False, False, False, False]
DONE after: 4.561900095723104e-05 seconds
CTRL: {'raw': '0xe', 'start': False, 'done': True, 'idle': True, 'ready': True}
DMA idle: [True, True, True, True]
First predictions: [6]


4.561900095723104e-05

In [10]:
print('COUNT readback:',ip.read(COUNT))
print('WINDOW_SIZE readback:',ip.read(WINDOW_SIZE_REG))
print('PRED LO:',hex(ip.read(PRED_LO)))
print('PRED HI:',hex(ip.read(PRED_HI)))
print('Expected pred:',hex(prediction_buffer.physical_address))

COUNT readback: 1
WINDOW_SIZE readback: 512
PRED LO: 0x15850000
PRED HI: 0x0
Expected pred: 0x15850000


## Full streamed FPGA test



In [11]:
def run_fpga_once(n,timeout_s=5.0,clear_predictions=True):
    assert 1<=n<=window_count

    if clear_predictions:
        prediction_buffer[:n]=0
        prediction_buffer.sync_to_device()

    for buf in input_buffers:
        buf.sync_to_device()

    program_hls_args(n)

    nbytes_per_bus=int(
        n *
        PACKETS_PER_WINDOW *
        np.dtype(np.uint64).itemsize
    )

    # Use the same timing boundary as the previous single-stream notebook:
    # start timer immediately before HLS AP_START, then launch DMA traffic,
    # and stop when the HLS core asserts AP_DONE.
    t0=time.perf_counter()

    ip.write(CTRL,AP_START)

    for dma,buf in zip(dmas,input_buffers):
        dma.sendchannel.transfer(
            buf,
            nbytes=nbytes_per_bus
        )

    while (ip.read(CTRL)&AP_DONE)==0:
        if time.perf_counter()-t0>timeout_s:
            raise TimeoutError(
                'HLS core did not assert AP_DONE; reload overlay before retrying.'
            )

    elapsed=time.perf_counter()-t0

    for dma in dmas:
        dma.sendchannel.wait()

    return elapsed


# Warm-up once.
_=run_fpga_once(window_count,timeout_s=5.0)

# Measured run.
fpga_s=run_fpga_once(window_count,timeout_s=5.0)

prediction_buffer.sync_from_device()
fpga_predictions=np.asarray(prediction_buffer,dtype=np.int32).copy()

print(f'Window size:                 {WINDOW_SIZE}')
print(f'FPGA 4-bus chained invoke:  {fpga_s:.9f} s')
print(f'FPGA average:               {fpga_s/window_count*1e6:.6f} us/window')
print(f'FPGA throughput:            {window_count/fpga_s:.2f} windows/s')


Window size:                 512
FPGA 4-bus chained invoke:  0.012135902 s
FPGA average:               2.022650 us/window
FPGA throughput:            494400.83 windows/s


## Accuracy and implementation agreement


In [12]:
fpga_accuracy=accuracy_score(y_test,fpga_predictions)
agreement_q18=float(np.mean(fpga_predictions==q18_predictions))
agreement_float=float(np.mean(fpga_predictions==folded_predictions))

multiclass_metrics,class_metric_table=classification_metrics(
    y_test,
    fpga_predictions,
    labels=np.arange(NUM_CLASSES),
    target_names=class_names
)

fault_metrics=fault_detection_metrics(
    y_test,
    fpga_predictions,
    normal_class=0
)

fpga_balanced_accuracy=multiclass_metrics['balanced_accuracy']
fpga_macro_precision=multiclass_metrics['macro_precision']
fpga_macro_recall=multiclass_metrics['macro_recall']
fpga_macro_f1=multiclass_metrics['macro_f1']
fpga_weighted_precision=multiclass_metrics['weighted_precision']
fpga_weighted_recall=multiclass_metrics['weighted_recall']
fpga_weighted_f1=multiclass_metrics['weighted_f1']

fpga_fault_precision=fault_metrics['fault_precision']
fpga_fault_recall=fault_metrics['fault_recall']
fpga_fault_f1=fault_metrics['fault_f1']

print(f'FPGA accuracy:                  {fpga_accuracy:.6f}')
print(f'FPGA balanced accuracy:         {fpga_balanced_accuracy:.6f}')
print(f'FPGA macro precision:           {fpga_macro_precision:.6f}')
print(f'FPGA macro recall:              {fpga_macro_recall:.6f}')
print(f'FPGA macro F1:                  {fpga_macro_f1:.6f}')
print(f'FPGA weighted precision:        {fpga_weighted_precision:.6f}')
print(f'FPGA weighted recall:           {fpga_weighted_recall:.6f}')
print(f'FPGA weighted F1:               {fpga_weighted_f1:.6f}')
print()
print('Binary Normal-vs-Fault metrics:')
print(f'  Fault precision:              {fpga_fault_precision:.6f}')
print(f'  Fault recall:                 {fpga_fault_recall:.6f}')
print(f'  Fault F1:                     {fpga_fault_f1:.6f}')
print(
    '  TP/FP/FN/TN:',
    fault_metrics['fault_tp'],
    fault_metrics['fault_fp'],
    fault_metrics['fault_fn'],
    fault_metrics['fault_tn']
)
print()
print(f'FPGA/Q18 prediction agreement:  {agreement_q18:.6f}')
print(f'FPGA/float-folded agreement:    {agreement_float:.6f}')
print('FPGA vs Q18 mismatches:',int(np.sum(fpga_predictions!=q18_predictions)))
print('Ground-truth errors:',int(np.sum(fpga_predictions!=y_test)))

print('\nPer-class precision / recall / F1:')
display(class_metric_table)

print('\nFull classification report:')
print(classification_report(
    y_test,
    fpga_predictions,
    labels=np.arange(NUM_CLASSES),
    target_names=class_names,
    digits=5
))

cm=confusion_matrix(
    y_test,
    fpga_predictions,
    labels=np.arange(NUM_CLASSES)
)
display(pd.DataFrame(cm,index=class_names,columns=class_names))

metrics_summary=pd.DataFrame([{
    'raw_sample_bits':RAW_SAMPLE_BITS,
    'window_size':WINDOW_SIZE,
    'accuracy':fpga_accuracy,
    'balanced_accuracy':fpga_balanced_accuracy,
    'macro_precision':fpga_macro_precision,
    'macro_recall':fpga_macro_recall,
    'macro_f1':fpga_macro_f1,
    'weighted_precision':fpga_weighted_precision,
    'weighted_recall':fpga_weighted_recall,
    'weighted_f1':fpga_weighted_f1,
    'fault_precision':fpga_fault_precision,
    'fault_recall':fpga_fault_recall,
    'fault_f1':fpga_fault_f1,
    'q18_agreement':agreement_q18,
    'float_folded_agreement':agreement_float,
}])

display(metrics_summary)
metrics_summary.to_csv(
    f'fpga_metrics_w{WINDOW_SIZE}_raw{RAW_SAMPLE_BITS}b.csv',
    index=False
)


FPGA accuracy:                  0.890500
FPGA balanced accuracy:         0.890500
FPGA macro precision:           0.890878
FPGA macro recall:              0.890500
FPGA macro F1:                  0.890064
FPGA weighted precision:        0.890878
FPGA weighted recall:           0.890500
FPGA weighted F1:               0.890064

Binary Normal-vs-Fault metrics:
  Fault precision:              0.998705
  Fault recall:                 1.000000
  Fault F1:                     0.999352
  TP/FP/FN/TN: 5400 7 0 593

FPGA/Q18 prediction agreement:  1.000000
FPGA/float-folded agreement:    0.999167
FPGA vs Q18 mismatches: 0
Ground-truth errors: 657

Per-class precision / recall / F1:


,class,precision,recall,f1_score,support
0,Normal,1.000000,0.988333,0.994132,600
1,B007,0.856240,0.903333,0.879157,600
2,IR007,0.941824,0.998333,0.969256,600
3,OR007,0.970297,0.980000,0.975124,600
4,B014,0.781726,0.770000,0.775819,600
5,IR014,0.934087,0.826667,0.877100,600
6,OR014,0.872274,0.933333,0.901771,600
7,B021,0.691400,0.683333,0.687343,600
8,IR021,0.916388,0.913333,0.914858,600
9,OR021,0.944541,0.908333,0.926083,600



Full classification report:
              precision     recall   f1-score   support

Normal          1.00000    0.98833    0.99413       600
B007            0.85624    0.90333    0.87916       600
IR007           0.94182    0.99833    0.96926       600
OR007           0.97030    0.98000    0.97512       600
B014            0.78173    0.77000    0.77582       600
IR014           0.93409    0.82667    0.87710       600
OR014           0.87227    0.93333    0.90177       600
B021            0.69140    0.68333    0.68734       600
IR021           0.91639    0.91333    0.91486       600
OR021           0.94454    0.90833    0.92608       600

accuracy                              0.89050      6000
macro avg       0.89088    0.89050    0.89006      6000
weighted avg    0.89088    0.89050    0.89006      6000


,Normal,B007,IR007,OR007,B014,IR014,OR014,B021,IR021,OR021
Normal,593,0,0,0,5,0,2,0,0,0
B007,0,542,0,0,0,0,0,58,0,0
IR007,0,0,599,0,0,0,0,0,1,0
OR007,0,0,0,588,0,0,0,0,12,0
B014,0,39,10,0,462,27,0,55,2,5
IR014,0,0,0,0,63,496,4,37,0,0
OR014,0,5,0,0,0,2,560,33,0,0
B021,0,47,7,0,55,5,76,410,0,0
IR021,0,0,9,16,0,0,0,0,548,27
OR021,0,0,11,2,6,1,0,0,35,545


,raw_sample_bits,window_size,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,fault_precision,fault_recall,fault_f1,q18_agreement,float_folded_agreement
0,12,512,0.8905,0.8905,0.890878,0.8905,0.890064,0.890878,0.8905,0.890064,0.998705,1.0,0.999352,1.0,0.999167


## Repeated timing for a stable streamed-input result


In [13]:
def repeated_fpga_timing(n=None,repeats=30,warmup=5):
    if n is None:
        n=window_count

    assert 1<=n<=window_count

    for buf in input_buffers:
        buf.sync_to_device()

    for _ in range(warmup):
        run_fpga_once(
            n,
            timeout_s=5.0,
            clear_predictions=False
        )

    times=[]

    for _ in range(repeats):
        times.append(
            run_fpga_once(
                n,
                timeout_s=5.0,
                clear_predictions=False
            )
        )

    times=np.asarray(times)

    result={
        'architecture':'4x64-bit max-packed chained MAV->Peak->ZC',
        'raw_sample_bits':RAW_SAMPLE_BITS,
        'window_size':WINDOW_SIZE,
        'samples_per_bus':SAMPLES_PER_BUS,
        'samples_per_packet':SAMPLES_PER_PACKET,
        'packets_per_window':PACKETS_PER_WINDOW,
        'last_packet_valid_samples':LAST_PACKET_VALID_SAMPLES,
        'windows':n,
        'repeats':repeats,
        'median_s':float(np.median(times)),
        'mean_s':float(np.mean(times)),
        'std_s':float(np.std(times)),
        'median_us_per_window':float(np.median(times)/n*1e6),
        'median_windows_per_s':float(n/np.median(times)),
        'fpga_accuracy':float(fpga_accuracy),
        'balanced_accuracy':float(fpga_balanced_accuracy),
        'macro_precision':float(fpga_macro_precision),
        'macro_recall':float(fpga_macro_recall),
        'macro_f1':float(fpga_macro_f1),
        'weighted_precision':float(fpga_weighted_precision),
        'weighted_recall':float(fpga_weighted_recall),
        'weighted_f1':float(fpga_weighted_f1),
        'fault_precision':float(fpga_fault_precision),
        'fault_recall':float(fpga_fault_recall),
        'fault_f1':float(fpga_fault_f1),
        'q18_agreement':float(agreement_q18)
    }

    return result


timing=repeated_fpga_timing()
print(timing)


{'architecture': '4x64-bit max-packed chained MAV->Peak->ZC', 'raw_sample_bits': 12, 'window_size': 512, 'samples_per_bus': 5, 'samples_per_packet': 20, 'packets_per_window': 26, 'last_packet_valid_samples': 12, 'windows': 6000, 'repeats': 30, 'median_s': 0.01172498799951427, 'mean_s': 0.011735067766494467, 'std_s': 2.4448987850368723e-05, 'median_us_per_window': 1.9541646665857118, 'median_windows_per_s': 511727.6026421999, 'fpga_accuracy': 0.8905, 'balanced_accuracy': 0.8905000000000001, 'macro_precision': 0.8908776066586321, 'macro_recall': 0.8905000000000001, 'macro_f1': 0.8900643090048914, 'weighted_precision': 0.8908776066586322, 'weighted_recall': 0.8905000000000001, 'weighted_f1': 0.8900643090048914, 'fault_precision': 0.9987053819123358, 'fault_recall': 1.0, 'fault_f1': 0.9993522716757658, 'q18_agreement': 1.0}
